Create the base list of the repos from URLs

In [23]:
import pandas as pd
import os

# === CONFIGURATION ===
INPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Sorted_URL_List.csv"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos.csv"

# === LOAD CSV ===
df = pd.read_csv(INPUT_CSV)

# === EXTRACT COMPONENTS IN LOWERCASE ===
df['github_url'] = df['github_url'].astype(str)
df['username'] = df['github_url'].apply(lambda x: x.split('/')[-2].lower())

# ✅ Robust: get last path part and remove trailing '.git' only
df['project_name'] = df['github_url'].apply(
    lambda x: x.split('/')[-1].lower().removesuffix('.git')
)

# ✅ Combine for full name with double underscore
df['full_name'] = df['username'] + '.' + df['project_name']

# Reset index to make it a column
df = df.reset_index()

# === REORDER: index + clone_url first ===
df_clean = df[['index', 'github_url', 'username', 'project_name', 'full_name']]
df_clean.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Cleaned repo list saved to: {OUTPUT_CSV}")


✅ Cleaned repo list saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos.csv


Aggregate YMLs and Builds

In [24]:
import pandas as pd
import os

# === CONFIGURATION ===
base_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis"

# === FILE PATHS ===
yml_csv = os.path.join(base_dir, "3.1_YML_List_ShallowC.csv")
yml_output_csv = os.path.join(base_dir, "3.1_YML_List_ShallowC_Aggregated.csv")

gradle_csv = os.path.join(base_dir, "3.2_Gradle_List_ShallowC.csv")
gradle_output_csv = os.path.join(base_dir, "3.2_Gradle_List_ShallowC_BUILD.csv")

# === LOAD ===
df_yml = pd.read_csv(yml_csv)
df_gradle = pd.read_csv(gradle_csv)

# === AGGREGATE YML ===
df_yml_agg = df_yml.groupby('full_name').agg({
    'ci_platform': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'test_type': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'matched_keywords': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'unit_test': 'any',
    'instrumentation_test': 'any',
    'full_name': 'count'  # temp for counting rows
}).rename(columns={
    'unit_test': 'unit_test_ci',
    'instrumentation_test': 'instr_test_ci',
    'ci_platform': 'ci_platform_yml',
    'full_name': 'NBR_YAML'
}).reset_index()

# === EXPORT YML ===
df_yml_agg.to_csv(yml_output_csv, index=False)
print(f"✅ Aggregated YML file saved to: {yml_output_csv}")

# === AGGREGATE GRADLE (including ci_platform_build) ===
df_gradle_agg = df_gradle.groupby('full_name').agg({
    'has_local_unit_test': 'any',
    'has_local_instrumentation_test': 'any',
    'ci_platform_build': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'full_name': 'count'  # for counting rows
}).rename(columns={
    'has_local_unit_test': 'unit_test_local',
    'has_local_instrumentation_test': 'instr_test_local',
    'ci_platform_build': 'ci_platform_build',
    'full_name': 'NBR_GRADLE'
}).reset_index()

# === EXPORT GRADLE ===
df_gradle_agg.to_csv(gradle_output_csv, index=False)
print(f"✅ Aggregated Gradle BUILD file saved to: {gradle_output_csv}")


✅ Aggregated YML file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.1_YML_List_ShallowC_Aggregated.csv
✅ Aggregated Gradle BUILD file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.2_Gradle_List_ShallowC_BUILD.csv


Merge 7.1 YML and Build to 7.4

In [27]:
import pandas as pd
import os

# === CONFIG ===
base_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis"

# ✅ Make sure this matches your true base file name:
base_csv = os.path.join(base_dir, "3.4_Total_Repos.csv")  # use your actual name
yml_agg_csv = os.path.join(base_dir, "3.1_YML_List_ShallowC_Aggregated.csv")
gradle_agg_csv = os.path.join(base_dir, "3.2_Gradle_List_ShallowC_BUILD.csv")
output_csv = os.path.join(base_dir, "3.4_Total_Repos_YML_Merged.csv")

# === LOAD ===
df_base = pd.read_csv(base_csv)
df_yml_agg = pd.read_csv(yml_agg_csv)
df_gradle_agg = pd.read_csv(gradle_agg_csv)

# === NORMALIZE KEYS ===
df_base['full_name'] = df_base['full_name'].astype(str).str.strip().str.lower()
df_yml_agg['full_name'] = df_yml_agg['full_name'].astype(str).str.strip().str.lower()
df_gradle_agg['full_name'] = df_gradle_agg['full_name'].astype(str).str.strip().str.lower()

# === MERGE BASE + YAML ===
df_merged = df_base.merge(
    df_yml_agg[['full_name', 'ci_platform_yml', 'test_type', 'matched_keywords', 'unit_test_ci', 'instr_test_ci', 'NBR_YAML']],
    on='full_name',
    how='left'
)

# === MERGE RESULT + GRADLE ===
df_merged = df_merged.merge(
    df_gradle_agg[['full_name', 'unit_test_local', 'instr_test_local', 'ci_platform_build', 'NBR_GRADLE']],
    on='full_name',
    how='left'
)

# === SAVE ===
df_merged.to_csv(output_csv, index=False)
print(f"✅ Final merged file saved to: {output_csv}")

print(f"✅ Base rows: {len(df_base)}")
print(f"✅ YAML agg rows: {len(df_yml_agg)}")
print(f"✅ Gradle agg rows: {len(df_gradle_agg)}")
print(f"✅ Rows with YAML info: {df_merged['ci_platform_yml'].notna().sum()}")
print(f"✅ Rows with Gradle info: {df_merged['unit_test_local'].notna().sum()}")


✅ Final merged file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos_YML_Merged.csv
✅ Base rows: 3582
✅ YAML agg rows: 3466
✅ Gradle agg rows: 1631
✅ Rows with YAML info: 3465
✅ Rows with Gradle info: 1631


appending metadata

In [34]:
import pandas as pd
import os

# === CONFIG ===
base_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis"
metadata_csv = os.path.join(base_dir, "Project_Metadata.csv")
base_final_csv = os.path.join(base_dir, "3.4_Total_Repos_YML_Merged.csv")
output_final_csv = os.path.join(base_dir, "3.4_Total_Repos_YML_Merged_Metadata.csv")

# === LOAD FILES ===
df_meta = pd.read_csv(metadata_csv)
df_base = pd.read_csv(base_final_csv)

# === RENAME AND NORMALIZE METADATA full_name ===
df_meta = df_meta.rename(columns={'full_name': 'full_name_metadata'})
df_meta['full_name'] = df_meta['full_name_metadata'].astype(str).str.replace("/", ".")
df_meta['full_name'] = df_meta['full_name'].str.strip().str.lower()

# === NORMALIZE BASE full_name ===
df_base['full_name'] = df_base['full_name'].astype(str).str.strip().str.lower()

# === SELECT METADATA COLUMNS ===
metadata_columns = [
    'full_name', 'language', 'license', 'created_at', 'updated_at', 'last_commit_date',
    'stars', 'forks', 'watchers', 'open_issues', 'size'
]
df_meta_selected = df_meta[metadata_columns]

# === MERGE ===
df_merged = df_base.merge(df_meta_selected, on='full_name', how='left')

# === SAVE MERGED FILE ===
df_merged.to_csv(output_final_csv, index=False)
print(f"✅ Final merged file with metadata saved to: {output_final_csv}")
print(f"✅ Base rows: {len(df_base)}")
print(f"✅ Metadata rows: {len(df_meta)}")
print(f"✅ Matched rows with metadata: {df_merged['language'].notna().sum()}")


KeyError: "['last_commit_date', 'stars', 'forks', 'watchers', 'open_issues'] not in index"

adding the number of contributos and commits

In [39]:
import pandas as pd
import os

# === CONFIGURATION ===
base_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis"

main_file = os.path.join(base_dir, "3.4_Total_Repos_YML_Merged.csv")
meta_file = os.path.join(base_dir, "Project_Metadata.csv")
output_file = os.path.join(base_dir, "3.4_Total_Repos_YML_Merged_Metadata_ByURL.csv")

# === LOAD CSV FILES ===
df_main = pd.read_csv(main_file)
df_meta = pd.read_csv(meta_file)

# === NORMALIZE JOIN KEYS ===
df_main['github_url'] = df_main['github_url'].astype(str).str.strip().str.lower()
df_meta['html_url'] = df_meta['html_url'].astype(str).str.strip().str.lower()

# === SELECT COLUMNS TO MERGE ===
columns_to_add = [
    'html_url', 'created_at', 'size', 'stargazers_count', 'updated_at',
    'pushed_at', 'forks_count', 'watchers_count', 'contributors',
    'pull_requests', 'commits_GitAPI'
]
df_meta_subset = df_meta[columns_to_add]

# === MERGE (LEFT JOIN) ===
df_merged = df_main.merge(
    df_meta_subset,
    left_on='github_url',
    right_on='html_url',
    how='left'
)

# === DROP REDUNDANT JOIN KEY COLUMN IF NEEDED ===
df_merged.drop(columns=['html_url'], inplace=True)

# === SAVE OUTPUT ===
df_merged.to_csv(output_file, index=False)
print(f"✅ Merged file saved to: {output_file}")
print(f"✅ Original main file rows: {len(df_main)}")
print(f"✅ Rows with matched metadata: {df_merged['created_at'].notna().sum()}")


✅ Merged file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos_YML_Merged_Metadata_ByURL.csv
✅ Original main file rows: 3582
✅ Rows with matched metadata: 3463


Creating General CI Platform and Unit / instr testing

In [41]:
import pandas as pd

# === 1) File paths ===
input_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos_YML_Merged_Metadata_ByURL.csv"
output_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos_YML_Merged_Metadata_ByURL_Updated.csv"

# === 2) Load CSV ===
df = pd.read_csv(input_path)

# === 3) Create 'CI Platform' ===
df['CI Platform'] = df[['ci_platform_yml', 'ci_platform_build']] \
    .fillna('') \
    .agg(','.join, axis=1) \
    .str.replace(r',+', ',', regex=True) \
    .str.strip(',')

# === 4) Use robust boolean logic without fillna ===
df['Unit_Test(CI or Local)'] = (df['unit_test_ci'] == True) | (df['unit_test_local'] == True)
df['Instr_Test(CI or Local)'] = (df['instr_test_ci'] == True) | (df['instr_test_local'] == True)

# === 5) Save ===
df.to_csv(output_path, index=False)
print(f"✅ Updated file saved to: {output_path}")


✅ Updated file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\3.4_Total_Repos_YML_Merged_Metadata_ByURL_Updated.csv
